# 11. Learning-Rate Range Test（spec §14.2）

## 学習目標
- 学習率を指数的に上げる短いrunで、loss・勾配ノルム・update比のLR依存を測る
- 不安定化点を検出し、推奨LRを根拠つきで自動提案する

## 数式
$$\eta_t = \eta_{\min}\left(\frac{\eta_{\max}}{\eta_{\min}}\right)^{t/T}$$
推奨LR = 「平滑化lossが最も急降下するLR」÷10（発散前の安全余裕）。

In [1]:
import warnings
warnings.filterwarnings("ignore")
import json, math
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "plotly_mimetype"
from jp_llm_lab.utils.io import repo_root, load_json, read_jsonl
ROOT = repo_root()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
M2_RUN = ROOT / "experiments/runs/m2_model_m_classical_seed42"
CAL = ROOT / "experiments/calibrations"

In [2]:
data = load_json(CAL / "lr_range.json")
recs = data["records"]
lrs = [r["lr"] for r in recs]
fig = go.Figure()
fig.add_trace(go.Scatter(x=lrs, y=[r["loss"] for r in recs], mode="lines+markers", name="loss",
                         line=dict(color="#1f77b4")))
fig.add_vline(x=data["suggested_lr"], line_dash="dash", line_color="#2ca02c",
              annotation_text=f'suggested {data["suggested_lr"]:.1e}')
if data["diverged_at_lr"]:
    fig.add_vline(x=data["diverged_at_lr"], line_dash="dot", line_color="#d62728",
                  annotation_text=f'diverged {data["diverged_at_lr"]:.1e}')
fig.update_xaxes(type="log", title="learning rate (log)")
fig.update_layout(title="LR range test: loss vs LR", yaxis_title="loss", template="plotly_white", height=400)
fig.show()
print(f"suggested_lr = {data['suggested_lr']:.2e}  (steepest {data['steepest_descent_lr']:.2e}, "
      f"diverged {data['diverged_at_lr']})")
print(data["logic"])

suggested_lr = 5.24e-03  (steepest 5.24e-02, diverged 1.0)
suggested = LR at steepest smoothed-loss descent / 10 (safety margin before divergence)


In [3]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=lrs, y=[r["grad_norm"] for r in recs], name="grad norm", line=dict(color="#d62728")))
fig.add_trace(go.Scatter(x=lrs, y=[r["update_ratio"] for r in recs], name="update ratio", yaxis="y2",
                         line=dict(color="#ff7f0e")))
fig.update_layout(title="LR vs 勾配ノルム / update比", template="plotly_white", height=380,
                  xaxis=dict(type="log", title="learning rate (log)"),
                  yaxis=dict(title="grad norm"), yaxis2=dict(title="update ratio", overlaying="y", side="right", type="log"))
fig.show()
print(f"Model M で実際に使った lr = 6e-4 は suggested {data['suggested_lr']:.1e} に対して"
      f" {'保守的' if 6e-4 < data['suggested_lr'] else '積極的'}")

Model M で実際に使った lr = 6e-4 は suggested 5.2e-03 に対して 保守的


## Observation / Interpretation / Caveat
- **Observation**: lossはLR増加とともに低下→ある点で急上昇（発散）。update比は単調増加。推奨LRは発散点の1–2桁下。
- **Interpretation**: 実運用の 6e-4 は推奨値より保守的で、安定性を優先した設定だったことが確認できる。範囲テストは「どこまで上げられるか」の上限を1回の短いrunで与える。
- **Caveat**: 単一バッチ列・1シードの short run。最適LRはバッチサイズ・スケジュール・モデルサイズに依存する。推奨ロジック（÷10）は経験則。

**次へ**: [12_batch_size_calibration](12_batch_size_calibration.ipynb)